# 08 · Build and train your own model

**Goals:** run a complete baseline, make an evidence-based improvement, and adapt the workflow to a new CSV. This notebook is independent of the others. See the [custom data guide](../guides/03-custom-data.md) for data preparation and task adaptations.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

torch.manual_seed(486)
torch.set_num_threads(1)  # Small examples run efficiently on a laptop.

root = Path.cwd()
if not (root / "data" / "observations.csv").exists():
    root = root.parent

data_path = root / "data" / "observations.csv"
assert data_path.exists(), "Start Jupyter in the repository root or notebooks directory."

frame = pd.read_csv(data_path)
features = ["feature_1", "feature_2", "feature_3"]
train = frame[frame["split"] == "train"]
validation = frame[frame["split"] == "validation"]
test = frame[frame["split"] == "test"]

scaler = StandardScaler().fit(train[features])  # Never fit on validation or test.

def tensors(rows):
    x = torch.tensor(scaler.transform(rows[features]), dtype=torch.float32)
    y = torch.tensor(rows["label"].to_numpy(), dtype=torch.long)
    return x, y

x_train, y_train = tensors(train)
x_val, y_val = tensors(validation)
x_test, y_test = tensors(test)


## Baseline

Read the cells carefully. Change architecture, optimizer, batch size, regularization, and epoch count one at a time. Use validation to compare choices. Record your choices and results; seeds reduce variation but do not ensure bit-for-bit reproduction everywhere.

In [ ]:
train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(x_val, y_val), batch_size=64, shuffle=False)

model = nn.Sequential(nn.Linear(3, 16), nn.ReLU(), nn.Linear(16, 2))

loss_fn = nn.CrossEntropyLoss()  # Inputs are raw logits; labels are integer indices.
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

def evaluate(model, loader):
    model.eval()
    total_loss, correct, count = 0.0, 0, 0
    with torch.no_grad():
        for x_batch, y_batch in loader:
            logits = model(x_batch)
            total_loss += loss_fn(logits, y_batch).item() * len(y_batch)
            correct += (logits.argmax(dim=1) == y_batch).sum().item()
            count += len(y_batch)
    return total_loss / count, correct / count

history = []
for epoch in range(1, 21):
    model.train()
    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()
        logits = model(x_batch)
        loss = loss_fn(logits, y_batch)
        loss.backward()
        optimizer.step()
    train_loss, train_accuracy = evaluate(model, DataLoader(TensorDataset(x_train, y_train), batch_size=64))
    val_loss, val_accuracy = evaluate(model, val_loader)
    history.append((epoch, train_loss, val_loss, train_accuracy, val_accuracy))
print(f"Epoch {epoch}: train loss={train_loss:.3f}, validation loss={val_loss:.3f}, validation accuracy={val_accuracy:.3f}")


In [ ]:
from sklearn.metrics import confusion_matrix

model.eval()

with torch.no_grad():
    val_predictions = model(x_val).argmax(dim=1).numpy()
print("Validation confusion matrix (rows: actual; columns: predicted):")
print(confusion_matrix(y_val.numpy(), val_predictions, labels=[0, 1]))
print("Majority-class validation accuracy:", max((y_val == 0).float().mean().item(), (y_val == 1).float().mean().item()))


## Your experiment

1. State a question and a baseline. Train and compare at least two choices using the validation set. Plot or tabulate results; report more than accuracy if class balance warrants it.

2. Once you have selected a final approach, evaluate once on the held-out test data. Explain limitations and what the test estimate represents.

3. Adapt this notebook to your CSV: change the path, feature names, label encoding, splits, input dimension, and output dimension as needed. Fit preprocessing only on training rows and check a batch's shapes and dtypes.

4. Save the model plus the information needed to preprocess and interpret new observations, following notebook 07. Record versions, split method, model settings, and random seeds.

For your own work, add code and explanation cells below. When sharing results, include how your data were collected and whether the observations are independent.

In [ ]:
# After choosing your final model using validation, change this to True.
run_final_test = False

if run_final_test:
    test_loader = DataLoader(TensorDataset(x_test, y_test), batch_size=64, shuffle=False)
    test_loss, test_accuracy = evaluate(model, test_loader)
    print(f"Final test loss: {test_loss:.3f}; test accuracy: {test_accuracy:.3f}")
else:
    print("Test evaluation is reserved until you select a final model.")
